In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import nltk
from nltk.corpus import wordnet as wn

In [3]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Method 1 (REJECTED)
match the wordnet synset using path similarity
however, path similarity can be calculated on same synsets.
thus, it doesn't work.

In [ ]:
def get_synset_by_similarity(word, context_synset_name):
    synsets = wn.synsets(word)
    context_synset = wn.synset(context_synset_name)
    if not synsets:
        return None

    # synset 간 path similarity 계산
    scored = []
    for s in synsets:
        sim = s.path_similarity(context_synset)
        print(sim, s.name(), context_synset)
        if sim:
            scored.append((s, sim))

    # 가장 높은 similarity 선택
    if scored:
        best = max(scored, key=lambda x: x[1])
        return best[0].name()
    else:
        return None

print(get_synset_by_similarity('hot', 'temperature.n.01'))  # → hot.a.01 (high temperature)
print(get_synset_by_similarity('hot', 'taste.n.01'))        # → hot.a.05 (spicy)

0.14285714285714285 hot.a.01 Synset('temperature.n.01')
0.14285714285714285 hot.s.02 Synset('temperature.n.01')
0.14285714285714285 hot.a.03 Synset('temperature.n.01')
0.14285714285714285 hot.s.04 Synset('temperature.n.01')
0.14285714285714285 hot.s.05 Synset('temperature.n.01')
0.14285714285714285 hot.s.06 Synset('temperature.n.01')
0.14285714285714285 blistering.s.03 Synset('temperature.n.01')
0.14285714285714285 hot.s.08 Synset('temperature.n.01')
0.14285714285714285 hot.s.09 Synset('temperature.n.01')
0.14285714285714285 hot.s.10 Synset('temperature.n.01')
0.14285714285714285 hot.s.11 Synset('temperature.n.01')
0.14285714285714285 hot.s.12 Synset('temperature.n.01')
0.14285714285714285 hot.s.13 Synset('temperature.n.01')
0.14285714285714285 hot.s.14 Synset('temperature.n.01')
0.14285714285714285 hot.s.15 Synset('temperature.n.01')
0.14285714285714285 hot.s.16 Synset('temperature.n.01')
0.14285714285714285 hot.s.17 Synset('temperature.n.01')
0.14285714285714285 hot.s.18 Synset('temp

# Method 2
using bert embedding

example)

definition of temperature.n.01 -> embedding 1

definition of hot.a.01 -> embedding 2

definition of hot.a.02 -> embedding 3, ...

=> compare embeddings.

Really works well.

In [4]:
from sentence_transformers import SentenceTransformer, util
from nltk.corpus import wordnet as wn

# 모델 로드 (최초 1회만)
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def get_synset_by_bert_similarity(word, context_word):
    synsets = wn.synsets(word)
    if not synsets:
        return None

    # context_word 임베딩
    context_emb = model.encode(context_word, convert_to_tensor=True)

    scored = []
    for s in synsets:
        definition_emb = model.encode(s.definition(), convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(context_emb, definition_emb).item()
        scored.append((s, similarity))

    best = max(scored, key=lambda x: x[1])
    return best[0].name(), best[0].definition(), best[1]

print(get_synset_by_bert_similarity('hot', 'temperature'))
print(get_synset_by_bert_similarity('hot', 'taste'))


('hot.a.01', 'used of physical heat; having a high or higher than desirable temperature or giving off heat or feeling or causing a sensation of heat or burning', 0.44773924350738525)
('hot.s.09', 'producing a burning sensation on the taste nerves', 0.4384460747241974)


In [6]:
%cd "/content/drive/MyDrive/NLP with Python"

/content/drive/MyDrive/NLP with Python


In [13]:
import os
import json
from tqdm import tqdm
from nltk.corpus import wordnet as wn
from sentence_transformers import SentenceTransformer, util

# BERT 모델 로드
model = SentenceTransformer('all-MiniLM-L6-v2')

done = [
    "bears_taco_menu.json",
    "byeoli_dali_menu.json",
    "campus_toast_menu.json",
    "insang_menu.json",
    "jesoon_menu.json",
    "little_hanoi_menu.json",
    "onigiri_and_lee_gyudong_menu.json",
    "pulbitmaru_menu.json",
    "rolling_pasta_menu.json",
    "the_big_lunch_box_menu_full.json",
    "well_chai_menu.json",
    "yeokjeon_menu.json"
]

# 처리할 JSON 파일 목록
file_names = [
    "subway_15cm_menu.json"
]

# context 문장 정의
context_sentences = {
    'temperature': 'This refers to the serving temperature of the food.',
    'taste': 'This describes the flavor profile of the food.',
    'ingredients': 'These are the ingredients used in the food.',
    'cuisine': 'This is the type of cuisine or cooking style.',
    'allergens': 'These are substances in food that may cause allergic reactions.'
}

# synset 추출 함수
def get_best_synset_by_bert(word, context_sentence):
    synsets = wn.synsets(word)
    if not synsets:
        return word
    context_emb = model.encode(context_sentence, convert_to_tensor=True)
    scored = []
    for s in synsets:
        def_emb = model.encode(s.definition(), convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(context_emb, def_emb).item()
        scored.append((s, similarity))
    best = max(scored, key=lambda x: x[1])
    return best[0].name()

# 경로 설정
input_dir = "./KAIST_MENUS/PROCESSED_JSON/"
output_dir = "./KAIST_MENUS/MATCHED_JSON/"
os.makedirs(output_dir, exist_ok=True)

# 각 파일 처리
for file_name in file_names:
    input_path = os.path.join(input_dir, file_name)
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in tqdm(data, desc=f"Processing {file_name}", leave=False):
        # temperature
        item['temperature'] = get_best_synset_by_bert(item['temperature'], context_sentences['temperature'])

        # taste
        taste_words = [t.strip() for t in item['taste']] if isinstance(item['taste'], list) else [t.strip() for t in item['taste'].split(',')]
        item['taste'] = [get_best_synset_by_bert(t, context_sentences['taste']) for t in taste_words]

        # ingredients
        item['ingredients'] = [get_best_synset_by_bert(ing, context_sentences['ingredients']) for ing in item['ingredients']]

        # allergens
        item['allergens'] = [get_best_synset_by_bert(a, context_sentences['allergens']) for a in item.get('allergens', [])]

    output_path = os.path.join(output_dir, file_name.replace(".json", "_with_synset.json"))
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

print("✅ Complete!")


✅ Complete!
